In [ ]:
import sys
import pickle
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

from src.data.transaction_utils import build_transactions

proc_dir = root / "data" / "processed" / "swat"
stream_dir = root / "data" / "stream" / "swat"

window_data_dir = stream_dir / "window_binary_data"
window_tx_dir = stream_dir / "window_transactions"
window_labeled_dir = stream_dir / "window_labeled_transactions"

window_data_dir.mkdir(parents=True, exist_ok=True)
window_tx_dir.mkdir(parents=True, exist_ok=True)
window_labeled_dir.mkdir(parents=True, exist_ok=True)

wid = 1064
exp_windows = pd.read_csv(stream_dir / "exp_windows_round1.csv")
row = exp_windows[exp_windows["window_id"] == wid].iloc[0]

start = int(row["start_idx"])
end = int(row["end_idx"])
attack_ratio = float(row["attack_ratio"])
lag = 2

# 1) 切窗口二值数据
df_binary = pd.read_csv(proc_dir / "X_filled_binary.csv")
local_df = df_binary.iloc[start:end + lag].reset_index(drop=True)
local_df.to_csv(window_data_dir / f"window_{wid}_binary.csv", index=False)

# 2) 构造 transactions
tx = build_transactions(local_df, lag=2)
with open(window_tx_dir / f"window_{wid}_transactions.pkl", "wb") as f:
    pickle.dump(tx, f)

# 3) 对齐标签并加 LABEL
y = pd.read_csv(proc_dir / "y_filled.csv").iloc[:, 0]
tx_labels = y.iloc[lag:].reset_index(drop=True)
labels_window = tx_labels.iloc[start:end].reset_index(drop=True)

tx_labeled = []
for one_tx, label in zip(tx, labels_window):
    new_tx = list(one_tx)
    new_tx.append("LABEL_ATTACK" if label == 1 else "LABEL_NORMAL")
    tx_labeled.append(new_tx)

with open(window_labeled_dir / f"window_{wid}_transactions_labeled.pkl", "wb") as f:
    pickle.dump(tx_labeled, f)

print("window_id:", wid)
print("attack_ratio:", attack_ratio)
print("transactions 数量:", len(tx))
print("标签分布:")
print(labels_window.value_counts())
print("saved:", window_data_dir / f"window_{wid}_binary.csv")
print("saved:", window_tx_dir / f"window_{wid}_transactions.pkl")
print("saved:", window_labeled_dir / f"window_{wid}_transactions_labeled.pkl")

In [ ]:
import pickle
import pandas as pd
from pathlib import Path
from mlxtend.preprocessing import TransactionEncoder

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"
window_labeled_dir = stream_dir / "window_labeled_transactions"
window_labeled_onehot_dir = stream_dir / "window_labeled_onehot"
window_labeled_onehot_dir.mkdir(parents=True, exist_ok=True)

wid = 1064

with open(window_labeled_dir / f"window_{wid}_transactions_labeled.pkl", "rb") as f:
    tx_labeled = pickle.load(f)

te = TransactionEncoder()
arr = te.fit(tx_labeled).transform(tx_labeled)
df_labeled_onehot = pd.DataFrame(arr, columns=te.columns_)

print(f"window {wid} -> labeled one-hot shape:", df_labeled_onehot.shape)
print(f"window {wid} -> LABEL_ATTACK exists:", "LABEL_ATTACK" in df_labeled_onehot.columns)
print(f"window {wid} -> LABEL_NORMAL exists:", "LABEL_NORMAL" in df_labeled_onehot.columns)
print(f"window {wid} -> LABEL_ATTACK sum:", int(df_labeled_onehot["LABEL_ATTACK"].sum()))
print(f"window {wid} -> LABEL_NORMAL sum:", int(df_labeled_onehot["LABEL_NORMAL"].sum()))

save_path = window_labeled_onehot_dir / f"window_{wid}_labeled_onehot.csv"
df_labeled_onehot.to_csv(save_path, index=False)

print("saved:", save_path)

In [ ]:
import pandas as pd
from pathlib import Path
from mlxtend.frequent_patterns import fpgrowth, association_rules

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"
window_labeled_onehot_dir = stream_dir / "window_labeled_onehot"
window_rule_dir = stream_dir / "window_label_rules"
window_rule_dir.mkdir(parents=True, exist_ok=True)

wid = 1064
df_labeled_onehot = pd.read_csv(window_labeled_onehot_dir / f"window_{wid}_labeled_onehot.csv")

freq_items_lbl = fpgrowth(
    df_labeled_onehot,
    min_support=0.1,
    use_colnames=True,
    max_len=2
)

rules_lbl = association_rules(
    freq_items_lbl,
    metric="confidence",
    min_threshold=0.6
)

rules_attack = rules_lbl[
    (rules_lbl["antecedents"].apply(len) == 1) &
    (rules_lbl["consequents"].apply(lambda x: x == frozenset({"LABEL_ATTACK"})))
].copy()

rules_normal = rules_lbl[
    (rules_lbl["antecedents"].apply(len) == 1) &
    (rules_lbl["consequents"].apply(lambda x: x == frozenset({"LABEL_NORMAL"})))
].copy()

rules_attack["antecedent_str"] = rules_attack["antecedents"].apply(lambda x: list(x)[0])
rules_normal["antecedent_str"] = rules_normal["antecedents"].apply(lambda x: list(x)[0])

rules_attack = rules_attack.sort_values(
    ["confidence", "lift", "support"],
    ascending=False
).reset_index(drop=True)

rules_normal = rules_normal.sort_values(
    ["confidence", "lift", "support"],
    ascending=False
).reset_index(drop=True)

print(f"window {wid} -> attack rules:", len(rules_attack))
print(rules_attack[["antecedent_str", "support", "confidence", "lift"]].head(10))

print(f"\nwindow {wid} -> normal rules:", len(rules_normal))
print(rules_normal[["antecedent_str", "support", "confidence", "lift"]].head(10))

rules_attack.to_csv(window_rule_dir / f"window_{wid}_rules_attack.csv", index=False)
rules_normal.to_csv(window_rule_dir / f"window_{wid}_rules_normal.csv", index=False)

print("\nsaved:", window_rule_dir / f"window_{wid}_rules_attack.csv")
print("saved:", window_rule_dir / f"window_{wid}_rules_normal.csv")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"
window_rule_dir = stream_dir / "window_label_rules"
window_pool_dir = stream_dir / "window_rule_pools"
window_pool_dir.mkdir(parents=True, exist_ok=True)

wid = 1064

rules_attack = pd.read_csv(window_rule_dir / f"window_{wid}_rules_attack.csv")
rules_normal = pd.read_csv(window_rule_dir / f"window_{wid}_rules_normal.csv")

# 平衡取样：攻击3条，正常3条
attack_top = rules_attack.head(3).copy()
normal_top = rules_normal.head(3).copy()

def confidence_to_weight(conf, eps=1e-6):
    conf = np.clip(conf, eps, 1 - eps)
    return float(np.log(conf / (1 - conf)))

def clipped_weight(conf, wmax=3.0):
    w = confidence_to_weight(conf)
    return float(np.clip(w, 0.0, wmax))

attack_top["consequent_str"] = "LABEL_ATTACK"
attack_top["formula"] = attack_top["antecedent_str"] + " => LABEL_ATTACK"
attack_top["weight"] = attack_top["confidence"].apply(lambda x: clipped_weight(x, wmax=3.0))
attack_top["target_label"] = 1

normal_top["consequent_str"] = "LABEL_NORMAL"
normal_top["formula"] = normal_top["antecedent_str"] + " => LABEL_NORMAL"
normal_top["weight"] = normal_top["confidence"].apply(lambda x: clipped_weight(x, wmax=3.0))
normal_top["target_label"] = 0

mixed_rule_pool = pd.concat([
    attack_top[["antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"]],
    normal_top[["antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"]],
], axis=0, ignore_index=True)

print(f"window {wid} mixed_rule_pool 数量:", len(mixed_rule_pool))
print("标签分布:")
print(mixed_rule_pool["target_label"].value_counts())
print(mixed_rule_pool[["formula", "confidence", "weight", "target_label"]])

save_path = window_pool_dir / f"window_{wid}_mixed_rule_pool.csv"
mixed_rule_pool.to_csv(save_path, index=False)
print("saved:", save_path)

In [ ]:
import sys
import importlib
import pickle
from pathlib import Path
import pandas as pd

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.state_utils as state_utils
importlib.reload(state_utils)

from src.rl.state_utils import build_initial_rule_state

stream_dir = root / "data" / "stream" / "swat"
window_pool_dir = stream_dir / "window_rule_pools"
window_state_dir = stream_dir / "window_rule_states"
window_state_dir.mkdir(parents=True, exist_ok=True)

wid = 1064
mixed_rule_pool = pd.read_csv(window_pool_dir / f"window_{wid}_mixed_rule_pool.csv")

window_state = build_initial_rule_state(mixed_rule_pool)

print(f"window {wid} -> num_rules:", window_state["num_rules"])
print(f"window {wid} -> weights:", window_state["weights"])
print(f"window {wid} -> target_labels:", window_state["target_labels"])

save_path = window_state_dir / f"window_{wid}_rule_state.pkl"
with open(save_path, "wb") as f:
    pickle.dump(window_state, f)

print("saved:", save_path)


In [ ]:

import sys
import importlib
import pickle
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.simple_env as simple_env
importlib.reload(simple_env)

from src.rl.simple_env import SimpleRuleEnv

stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"

wid = 1064

with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

env = SimpleRuleEnv(window_state, max_steps=1)

# 1) keep 一个攻击规则
env.reset()
_, r_attack_keep, _, info1 = env.step(rule_idx=0, action=0)

# 2) disable 一个攻击规则
env.reset()
_, r_attack_disable, _, info2 = env.step(rule_idx=0, action=1)

# 3) disable 一个正常规则（rule_idx=3 开始是正常规则）
env.reset()
_, r_normal_disable, _, info3 = env.step(rule_idx=3, action=1)

print("r_attack_keep:", r_attack_keep, info1)
print("r_attack_disable:", r_attack_disable, info2)
print("r_normal_disable:", r_normal_disable, info3)


In [ ]:
import sys
import importlib
import pickle
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.ac_model as ac_model
import src.rl.obs_utils as obs_utils
importlib.reload(ac_model)
importlib.reload(obs_utils)

from src.rl.ac_model import ActorCriticNet
from src.rl.obs_utils import build_state_vector

stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"
model_dir = root / "outputs" / "models"
model_dir.mkdir(parents=True, exist_ok=True)

wid = 1064
with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

active_mask = window_state["active_mask"]
weights = window_state["weights"]
rule_scores = window_state["rule_scores"]
target_labels = window_state["target_labels"]

# 构造监督数据：攻击规则->keep(0), 正常规则->disable(1)
X_list, y_list = [], []
for rule_idx in range(window_state["num_rules"]):
    state_vec = build_state_vector(
        active_mask, weights, rule_scores, rule_idx, target_labels
    )
    X_list.append(state_vec)
    y_list.append(0 if target_labels[rule_idx] == 1 else 1)

X = torch.tensor(np.array(X_list, dtype=np.float32))
y = torch.tensor(np.array(y_list, dtype=np.int64))

model_warm = ActorCriticNet(state_dim=12, action_dim=2, hidden_dim=64)
optimizer = torch.optim.Adam(model_warm.parameters(), lr=1e-3)

for epoch in range(300):
    logits, values = model_warm(X)
    cls_loss = F.cross_entropy(logits, y)

    optimizer.zero_grad()
    cls_loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        pred = logits.argmax(dim=1)
        acc = (pred == y).float().mean().item()
        print(f"epoch={epoch+1}, cls_loss={cls_loss.item():.6f}, acc={acc:.4f}")

with torch.no_grad():
    logits, _ = model_warm(X)
    pred = logits.argmax(dim=1)
    acc = (pred == y).float().mean().item()

save_path = model_dir / "window_1064_actor_warmstart.pth"
torch.save(model_warm.state_dict(), save_path)

print("final acc:", acc)
print("pred:", pred.tolist())
print("true:", y.tolist())
print("saved:", save_path)

In [ ]:
import sys
import importlib
import pickle
from pathlib import Path
import pandas as pd
import torch

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.ac_model as ac_model
import src.rl.simple_env as simple_env
import src.rl.action_utils as action_utils

importlib.reload(ac_model)
importlib.reload(simple_env)
importlib.reload(action_utils)

from src.rl.ac_model import ActorCriticNet
from src.rl.simple_env import SimpleRuleEnv
from src.rl.action_utils import ACTION_NAMES

stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"
window_pool_dir = stream_dir / "window_rule_pools"
model_dir = root / "outputs" / "models"

wid = 1064

with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

rule_pool_df = pd.read_csv(window_pool_dir / f"window_{wid}_mixed_rule_pool.csv")

model = ActorCriticNet(state_dim=12, action_dim=2, hidden_dim=64)
model.load_state_dict(torch.load(model_dir / "window_1064_actor_warmstart.pth", map_location="cpu"))
model.eval()

env = SimpleRuleEnv(window_state, max_steps=window_state["num_rules"])

state = env.reset()
records = []
total_reward = 0.0

for step in range(window_state["num_rules"]):
    rule_idx = step

    state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    logits, value = model(state_tensor)
    action = torch.argmax(logits, dim=-1).item()

    next_state, reward, done, info = env.step(rule_idx, action)
    total_reward += reward

    records.append({
        "rule_idx": rule_idx,
        "target_label": int(rule_pool_df.iloc[rule_idx]["target_label"]),
        "formula": rule_pool_df.iloc[rule_idx]["formula"],
        "action_name": ACTION_NAMES[action],
        "reward": reward
    })

    state = next_state
    if done:
        break

eval_df = pd.DataFrame(records)

print(eval_df[["rule_idx", "target_label", "action_name", "reward"]])
print("\n动作统计:")
print(eval_df["action_name"].value_counts())
print("\n按 target_label 分组统计:")
print(pd.crosstab(eval_df["target_label"], eval_df["action_name"]))
print("\ngreedy total_reward:", total_reward)

In [ ]:
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
log_dir = root / "outputs" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

attack_keep_rate = (
    ((eval_df["target_label"] == 1) & (eval_df["action_name"] == "keep")).sum()
    / (eval_df["target_label"] == 1).sum()
)

normal_disable_rate = (
    ((eval_df["target_label"] == 0) & (eval_df["action_name"] == "disable")).sum()
    / (eval_df["target_label"] == 0).sum()
)

selection_accuracy = (
    ((eval_df["target_label"] == 1) & (eval_df["action_name"] == "keep")).sum()
    + ((eval_df["target_label"] == 0) & (eval_df["action_name"] == "disable")).sum()
) / len(eval_df)

result_df = pd.DataFrame([{
    "window_id": 1064,
    "attack_keep_rate": attack_keep_rate,
    "normal_disable_rate": normal_disable_rate,
    "selection_accuracy": selection_accuracy,
    "greedy_total_reward": total_reward
}])

save_path = log_dir / "window_1064_metrics.csv"
result_df.to_csv(save_path, index=False)

print("saved:", save_path)
print(result_df)

In [ ]:
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
log_dir = root / "outputs" / "logs"

df_998 = pd.read_csv(log_dir / "window_998_metrics.csv")
df_1436 = pd.read_csv(log_dir / "window_1436_metrics.csv")
df_1064 = pd.read_csv(log_dir / "window_1064_metrics.csv")

summary_df = pd.concat([df_998, df_1436, df_1064], axis=0, ignore_index=True)
summary_df["window_type"] = "mixed"

save_path = log_dir / "mixed_windows_summary_v2.csv"
summary_df.to_csv(save_path, index=False)

print("saved:", save_path)
print(summary_df)

print("\nmean attack_keep_rate:", summary_df["attack_keep_rate"].mean())
print("mean normal_disable_rate:", summary_df["normal_disable_rate"].mean())
print("mean selection_accuracy:", summary_df["selection_accuracy"].mean())
print("mean greedy_total_reward:", summary_df["greedy_total_reward"].mean())

In [ ]:
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
log_dir = root / "outputs" / "logs"
stream_dir = root / "data" / "stream" / "swat"

summary_df = pd.read_csv(log_dir / "mixed_windows_summary_v2.csv")
exp_windows = pd.read_csv(stream_dir / "exp_windows_round1.csv")

summary_with_ratio = summary_df.merge(
    exp_windows[["window_id", "attack_ratio"]],
    on="window_id",
    how="left"
)

summary_with_ratio = summary_with_ratio[[
    "window_id",
    "attack_ratio",
    "attack_keep_rate",
    "normal_disable_rate",
    "selection_accuracy",
    "greedy_total_reward",
    "window_type"
]]

save_path = log_dir / "mixed_windows_summary_with_ratio_v1.csv"
summary_with_ratio.to_csv(save_path, index=False)

print("saved:", save_path)
print(summary_with_ratio.sort_values("attack_ratio"))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
log_dir = root / "outputs" / "logs"
fig_dir = root / "outputs" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(log_dir / "mixed_windows_summary_with_ratio_v1.csv")

plt.figure(figsize=(6, 4))
plt.scatter(df["attack_ratio"], df["selection_accuracy"])

for _, row in df.iterrows():
    plt.text(row["attack_ratio"], row["selection_accuracy"], str(int(row["window_id"])))

plt.xlabel("Attack Ratio")
plt.ylabel("Selection Accuracy")
plt.title("Attack Ratio vs Selection Accuracy")
plt.tight_layout()

save_path = fig_dir / "attack_ratio_vs_selection_accuracy_v1.png"
plt.savefig(save_path, dpi=200)
plt.show()

print("saved:", save_path)
print(df[["window_id", "attack_ratio", "selection_accuracy"]])